# MedLoRA · 只评估 CPT 阶段的 adapter (不做 SFT), 与基座对比

数据源挂两个 kernel 的输出: `medlora-cpt-b` (B1 的 `cpt_pubmed_qlora_r16`) 和 `medlora-cpt-b2-iu-x-ray-image-text` (B2 的 `cpt_iu_qlora_r16`)。三张表各跑一遍, 约 1.5 h 一个。

In [ ]:
REPO_URL = "https://github.com/AugustLoo/MedLoRA.git"
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
!git clone -q $REPO_URL /kaggle/working/MedLoRA
%cd /kaggle/working/MedLoRA
!mkdir -p /kaggle/temp/raw && rm -rf data/raw && ln -s /kaggle/temp/raw data/raw
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob, os
ADAPTERS = {}
for cfg in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True):
    d = os.path.dirname(cfg)
    if d.rstrip('/').endswith('cpt_pubmed_qlora_r16'): ADAPTERS['cpt_only_b1'] = d
    if d.rstrip('/').endswith('cpt_iu_qlora_r16'): ADAPTERS['cpt_only_b2'] = d
print(ADAPTERS)
assert len(ADAPTERS) == 2, 'CPT adapters not found under /kaggle/input'

In [ ]:
!python data/download_slake.py | tail -3
!python data/download_pubmedqa.py | tail -2

In [ ]:
for tag, path in ADAPTERS.items():
    print('==', tag, path)
    !CUDA_VISIBLE_DEVICES=0 MODEL=$MODEL bash train/eval_all.sh {tag} {path} 2>&1 | grep -v -E "it/s\]|s/it\]"

In [ ]:
import json
base = json.load(open("results/baseline_2026-09-15.json"))
a = json.load(open("results/sft_A_2026-09-15.json"))
res = {t: {k: json.load(open(f"outputs/eval/{k}_{t}.json")) for k in ["slake", "textvqa", "pubmedqa"]} for t in ADAPTERS}
cols = [("base", base), ("B1-CPT", res["cpt_only_b1"]), ("B2-CPT", res["cpt_only_b2"]), ("A", a)]
def row(name, *v): print(f"{name:<22}" + "".join(f"{x:>9}" for x in v))
row("metric", *[c for c, _ in cols])
for k in ["closed_acc", "open_em", "open_recall", "open_f1"]:
    row("slake_" + k, *[x["slake"]["metrics"][k] for _, x in cols])
row("textvqa_acc", *[x["textvqa"]["textvqa_acc"] for _, x in cols])
row("pubmedqa_acc", *[x["pubmedqa"]["accuracy"] for _, x in cols])
row("pubmedqa_macro_f1", *[x["pubmedqa"]["macro_f1"] for _, x in cols])
for c, x in cols: print(c, "pubmedqa pred_dist:", x["pubmedqa"]["pred_dist"], "unparsed:", x["pubmedqa"].get("unparsed"))
print("RESULTS_JSON", json.dumps(res))